# Supplementary figure — Temperature curves for all stimulus seeds

In [1]:
%%capture
from pathlib import Path

if Path.cwd().name == "notebooks_altair":
    %cd ..

%load_ext autoreload
%autoreload 2

In [2]:
%%capture
import tomllib
from pathlib import Path

import numpy as np
import polars as pl

from src.experiments.measurement.stimulus_generator import StimulusGenerator
from src.plots_altair import plot_stimulus_seed_grid, style_figure

In [3]:
configuration_path = Path("src/experiments/measurement/measurement_config.toml")
with configuration_path.open("rb") as file:
    stimulus_config = dict(tomllib.load(file)["stimulus"])

stimulus_config.update(
    {
        "temperature_baseline": 44.5,
        "temperature_range": 3,
        "sample_rate": 2,
    }
)
stimulus_frames = []
for seed in stimulus_config["seeds"]:
    generated = StimulusGenerator(stimulus_config, seed=seed)
    stimulus_frames.append(
        pl.DataFrame(
            {
                "seed": [seed] * len(generated.y),
                "time_s": np.arange(len(generated.y)) / generated.sample_rate,
                "temperature": generated.y,
            }
        )
    )
stimuli = pl.concat(stimulus_frames)

In [4]:
stimulus_seed_grid_chart = plot_stimulus_seed_grid(
    stimuli,
    columns=3,
    width=300,
    height=105,
    line_color="#17396b",
    line_width=2,
    panel_spacing=12,
    header_font_size=15,
    title=None,
)
style_figure(stimulus_seed_grid_chart)

alt.FacetChart(...)

In [5]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
figure_dir = Path(os.environ["FIGURE_DIR"])
figure_dir.mkdir(parents=True, exist_ok=True)
style_figure(stimulus_seed_grid_chart).save(
    figure_dir / "supplementary_temperature_curves.svg"
)